In [4]:
# Put import statements here
import os
# hide tensorflow info/warning logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import sys
import subprocess
from pathlib import Path
import json


# Local files/code

import src.data_preprocessing.text_preprocessing as text_pre
from src.agents.Visual_Agent import VisualModel
from src.util.logger import Logger
import config as config
import src.util.general as general_util
from src.data_preprocessing.TextTokenizer import TextTokenizer
from src.data_preprocessing.TextEmbedder import TextEmbedder
from data_preprocessing.USDA_processing import embed_USDA_data
import src.evaluation.visual_evaluator as visual_evaluator
import src.data_preprocessing.image_preprocessing as img_pre
from src.agents.NLP_Agent import AnswerRanker, build_answer_index, add_usda_records
import src.evaluation.text_evaluator as text_eval


In [ ]:

# Chunk and embed the USDA data

# make the text embedder
embedder= TextEmbedder("sentence-transformers/all-MiniLM-L6-v2")

CHUNK_METHOD= "word" # word or sentence
CHUNK_MAX_SIZE= 64
CHUNK_OVERLAP_AMOUNT= 16
#temp= config.Data.USDA.PLANT_SHEETS / "ABBA/ABBA_1.pdf"

# load and clean the text data however you need to
#text= text_pre.load_pdf_text_data(temp)
#text= text_pre.replace_urls(text, "url")
#text= text.replace("<", "")
#text= text.replace(">", "")

# chunk and encode
#records= embedder.chunk_and_encode(text, CHUNK_METHOD, CHUNK_MAX_SIZE, CHUNK_OVERLAP_AMOUNT)
# This takes ~30sec - 1.5min to run
# parse the USDA plant sheets, chunk them, vectorize them, and make organized records
# Also chunk/vectorize USDA json files that have relevant plant sheets
# NOTE: the PDF parsing library loves to spit out annoying logs
#       I've tried to supress them a million ways, but could not get it to stop
#       The logs are annoying but superficial
records= embed_USDA_data(
    config.Data.USDA.ROOT,
    embedder,
    CHUNK_METHOD,
    CHUNK_MAX_SIZE,
    CHUNK_OVERLAP_AMOUNT
)

# review records
for record in records:
    Logger.info(f"\n{record}")



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3010.15it/s]


MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error:

In [3]:
# Example of one of the records for one chunk of a plant
Logger.info(f"records type: {type(records)}")
Logger.info(f"records len: {len(records)}")
Logger.info(f"Example of a pdf chunk record: {records[0]}")
Logger.info(f"Example of a json record: \n\n{records[-1]}")

14:39:13 [INFO    ]: records type: <class 'list'>
14:39:13 [INFO    ]: records len: 19158
14:39:13 [INFO    ]: Example of a pdf chunk record: Text: Plant Guide Plant Materials url Plant Fact Sheet/G...
Embedding: [-0.03264537453651428, -0.08751830458641052, -0.013247602619230747, 0.05796762555837631, 0.07962250709533691, '...']
Metadata: {'code': 'ABAM', 'common name': 'Pacific silver fir', 'scientific name': 'Abies amabilis (Douglas ex Loudon) Douglas ex Forbes', 'source file': 'ABAM_1.pdf'}
14:39:13 [INFO    ]: Example of a json record: 

Text: symbol: ZOTE
scientific name: Zoysia tenuifolia Wi...
Embedding: [-0.09412891417741776, 0.06815403699874878, -0.02247413992881775, -0.04799898713827133, -0.014484964311122894, '...']
Metadata: {'code': 'ZOTE', 'common name': 'Mascarene grass', 'scientific name': 'Zoysia tenuifolia Willd. ex Thiele', 'source file': 'ZOTE'}


In [4]:


# preprocess (not tokenize) the training, testing, and validation datasets

plant_expert_vqa_TRAIN= text_pre.load_csv(config.Data.PlantExpertVQA.TRAIN_FILE)
plant_expert_vqa_TEST= text_pre.load_csv(config.Data.PlantExpertVQA.TEST_FILE)
plant_expert_vqa_VAL= text_pre.load_csv(config.Data.PlantExpertVQA.VALIDATION_FILE)

# training

text_pre.preprocess_dataframe(
    plant_expert_vqa_TRAIN,
    config.Data.PlantExpertVQA.TEXT_COLUMNS,
    config.Data.PlantExpertVQA.COLUMNS_TO_REMOVE,
    config.Data.PlantExpertVQA.NA_FILL,
    ["image_path"],
    config.Data.PlantExpertVQA.ROOT,
)

# testing
text_pre.preprocess_dataframe(
    plant_expert_vqa_TEST,
    config.Data.PlantExpertVQA.TEXT_COLUMNS,
    config.Data.PlantExpertVQA.COLUMNS_TO_REMOVE,
    config.Data.PlantExpertVQA.NA_FILL,
    ["image_path"],
    config.Data.PlantExpertVQA.ROOT,
)

#validation
text_pre.preprocess_dataframe(
    plant_expert_vqa_VAL,
    config.Data.PlantExpertVQA.TEXT_COLUMNS,
    config.Data.PlantExpertVQA.COLUMNS_TO_REMOVE,
    config.Data.PlantExpertVQA.NA_FILL,
    ["image_path"],
    config.Data.PlantExpertVQA.ROOT,
)


14:39:15 [DEBUG   ]: [load_csv] Successfully loaded /mnt/c/Users/codyr/Code/AI-572/PlantQA/data/PlantExpertVQA/data/train.csv into dataframe


/mnt/c/Users/codyr/Code/AI-572/PlantQA/src/data_preprocessing/text_preprocessing.py:48: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  data= pd.read_csv(csv_path)


14:39:15 [DEBUG   ]: [load_csv] Successfully loaded /mnt/c/Users/codyr/Code/AI-572/PlantQA/data/PlantExpertVQA/data/test.csv into dataframe
14:39:16 [DEBUG   ]: [load_csv] Successfully loaded /mnt/c/Users/codyr/Code/AI-572/PlantQA/data/PlantExpertVQA/data/val.csv into dataframe


In [5]:
# Testing the tokenizer and showing how to use it

# intialize the tokenizer
tokenizer= TextTokenizer("distilbert-base-uncased", 128)

# example of the passing in a single string to tokenize
inputs, attention_mask= tokenizer.encode_text("hello")
Logger.info(inputs)

# how to decode a single tensor output
output= tokenizer.decode(inputs)
Logger.info(output)

# encoding a list of strings
inputs, attention_mask= tokenizer.encode_text(["hello", "this is a test", "i am putting in multiple strings"])
Logger.info(inputs)

# decoding a list of tensors
output= tokenizer.batch_decode(inputs)
Logger.info(output)

# encoding a column of data
questions= plant_expert_vqa_TRAIN["question_text"]
inputs, attention_mask= tokenizer.encode_text(questions)

# decoding a all of those tensors
output= tokenizer.batch_decode(inputs)
Logger.info(f"first few decoded Tensors: \n\n{output[:10]}")


14:39:22 [INFO    ]: tf.Tensor([[ 101 7592  102]], shape=(1, 3), dtype=int64)
14:39:22 [INFO    ]: ['hello']
14:39:22 [INFO    ]: tf.Tensor(
[[ 101 7592  102    0    0    0    0    0]
 [ 101 2023 2003 1037 3231  102    0    0]
 [ 101 1045 2572 5128 1999 3674 7817  102]], shape=(3, 8), dtype=int64)
14:39:22 [INFO    ]: ['hello', 'this is a test', 'i am putting in multiple strings']


I0000 00:00:1790102361.646225     990 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13271 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4080 SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9


14:39:34 [INFO    ]: first few decoded Tensors: 

['what disease shown here', 'how severe gourd anthracnose infection', 'what led severe gourd anthracnose infection', 'gourd anthracnose treated 5 7 days ago when symptoms first appeared how leaf look now', 'without treatment what happen leaf over next 1 2 weeks', 'what immediate action needed severe gourd anthracnose', 'severe gourd anthracnose prevented', 'what disease shown here', 'how severe gourd anthracnose infection', 'what led severe gourd anthracnose infection']


In [8]:
traits = ["crop", "disease", "severity"]

train_img = VisualModel.one_row_per_image(plant_expert_vqa_TRAIN, traits)
val_img = VisualModel.one_row_per_image(plant_expert_vqa_VAL, traits)
test_img = VisualModel.one_row_per_image(plant_expert_vqa_TEST, traits)
classes = VisualModel.build_classes(train_img, traits)

train_ds = VisualModel.make_dataset(train_img, classes, config.Data.PlantExpertVQA.ROOT, training=True, use_mask=True)
val_ds = VisualModel.make_dataset(val_img, classes, config.Data.PlantExpertVQA.ROOT, use_mask=True)

In [ ]:
visual_model = VisualModel(classes=classes, use_mask=True)
visual_model.build()
visual_model.compile()
history_frozen = visual_model.fit(train_ds, val_ds, epochs=20)
visual_model.unfreeze_layers()
history_tuned = visual_model.fit(train_ds, val_ds, epochs=5)

visual_evaluator.plot_loss(history_frozen, history_tuned)
visual_evaluator.plot_accuracy(history_frozen, history_tuned)
visual_evaluator.plot_accuracy(history_frozen, history_tuned, heads=["crop", "disease"])

Epoch 1/20


I0000 00:00:1790102559.024271    1212 service.cc:152] XLA service 0x66ad7950 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1790102559.024320    1212 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4080 SUPER, Compute Capability 8.9
I0000 00:00:1790102560.241630    1212 cuda_dnn.cc:529] Loaded cuDNN version 92400
2026-09-22 14:42:45.690506: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 14:42:45.831397: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 14:42:46.386747: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal 

   2/3285 ━━━━━━━━━━━━━━━━━━━━ 5:32 101ms/step - crop_accuracy: 0.1094 - crop_loss: 3.6379 - disease_accuracy: 0.0000e+00 - disease_loss: 4.6826 - loss: 9.8641 - severity_accuracy: 0.3438 - severity_loss: 1.5436  

I0000 00:00:1790102571.335748    1212 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  99/3285 ━━━━━━━━━━━━━━━━━━━━ 4:19 81ms/step - crop_accuracy: 0.3991 - crop_loss: 2.4009 - disease_accuracy: 0.2387 - disease_loss: 3.1460 - loss: 6.9171 - severity_accuracy: 0.3942 - severity_loss: 1.3702

In [ ]:
import json
Path("./models/visual_models/9_22_classes.json").write_text(json.dumps(classes))
visual_model.save(path="./models/visual_models/9_22.keras")

In [ ]:
MODEL_DIR = Path("./models/visual_models")

vm = VisualModel.load(MODEL_DIR / "9_22.keras", MODEL_DIR/"9_22_classes.json")

In [ ]:
test_table, test_predictions = visual_evaluator.evaluate(vm, test_img, config.Data.PlantExpertVQA.ROOT, name="Test_VisualModel")
test_table

In [2]:
import pandas as pd
import src.agents.NLP_Agent
import importlib
import src.evaluation.text_evaluator

importlib.reload(src.agents.NLP_Agent)
importlib.reload(src.evaluation.text_evaluator)
from src.agents.NLP_Agent import AnswerRanker, build_answer_index, add_usda_records
index = build_answer_index(embedder, plant_expert_vqa_TRAIN)
ranker = AnswerRanker(index)
add_usda_records(index, records)

text_eval.inspect(ranker, plant_expert_vqa_VAL, count=3)
text_eval.evaluate_ranker(ranker, plant_expert_vqa_VAL, sample=200)

2026-09-22 21:29:23.743850: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-22 21:29:23.862963: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790126963.907784     947 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790126963.921282     947 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1790126964.023511     947 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

NameError: name 'embedder' is not defined